# Colour Brings People Together
### Part C · Creating an empirically grounded colour palette

In Part B you found what speakers of several languages call a handful of SDG colours. Now reverse the direction. You will give those **names** to an AI system, ask it for the colour each name stands for, and check its answers against where speakers actually centre each name. Where the AI's colour diverges, you replace it with the Colour Spinner's best example, and you finish with a small palette you can defend with evidence.

| Step | You do | Time |
|---|---|---|
| 1 | Load your names from Part B | 5 min |
| 2 | Ask an AI system for the best example of each name | 10 min |
| 3 | Compare its colours with the empirical best examples | 10 min |
| 4 | Replace divergent colours and check the result still works for every viewer | 15 min |
| 5 | Export the palette and prepare your finding | 5 min |
| | Plenary discussion | 15 min |

Run each cell with **Shift + Enter** and change only the grey form fields.

In [ ]:
#@title Run me first: load the masterclass toolkit
#@markdown The colour maths, the Colour Spinner connection and the charts live in `colour_masterclass.py`, published next to these notebooks.
TOOLKIT_URL = "https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/colour_masterclass.py"  #@param {type:"string"}
import os, urllib.request
if not os.path.exists("colour_masterclass.py"):
    try:
        urllib.request.urlretrieve(TOOLKIT_URL, "colour_masterclass.py")
    except Exception as e:
        raise RuntimeError("Could not download the toolkit. Check your internet connection, or upload "
                           "colour_masterclass.py in Colab's Files panel (folder icon, left).") from e
from colour_masterclass import *

## 1 · Load your names from Part B

Upload the `partB_results.csv` you downloaded at the end of Part B. If you lost it, choose *Type them in* and list a few colour names per language instead; the notebook will look up their best examples.

In [ ]:
#@title Where your names come from
NAMES_FROM = "Upload partB_results.csv"  #@param ["Upload partB_results.csv", "Type them in"]
BACKEND = "Colour Spinner API"  #@param ["Colour Spinner API", "Copied from the browser"]
#@markdown *Copied from the browser* only matters if you type names in: then add each name's best example to `SHEET` below.

In [ ]:
TYPED_NAMES = """
language, name
# American English, Red          ← one row per name, only if you chose "Type them in"
"""

SHEET = """
hex, language, name, confidence, best_example
"""

In [ ]:
names = load_names(NAMES_FROM, TYPED_NAMES)
LANGUAGES = list(dict.fromkeys(names.language))
show_names(names)

## 2 · Ask the AI system for the best example of each name

The prompt asks for the colour that a native speaker would consider the **best example** of each name: the most typical instance, not the average and not a brand colour. As in Part B, Colab's built-in Gemini is used for demonstration, and you can use any AI you use regularly instead: choose *Paste the reply from my own AI* in the form below, and the next cell gives you the prompt and a box to paste the reply into. Use the same AI system as in Part B so the two parts can be compared.

In [ ]:
#@title Which AI?
AI_PROVIDER = "Gemini built into Colab (no key needed)"  #@param ["Gemini built into Colab (no key needed)", "Paste the reply from my own AI", "Anthropic", "OpenAI-compatible"]
AI_MODEL = ""  #@param {type:"string"}
AI_BASE_URL = ""  #@param {type:"string"}
#@markdown **Gemini built into Colab** needs nothing more: just run the cells below.
#@markdown **Your own AI** (ChatGPT, Claude, Copilot, …): choose *Paste the reply from my own AI*; the next cell gives you the prompt and a box to paste the reply into.
#@markdown *Anthropic* and *OpenAI-compatible* call an API with your own key from Colab ▸ Secrets (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) and a model name. Leave AI_MODEL empty for Colab's default Gemini.

In [ ]:
items = "\n".join(f"- {r.language}: {plain_with_roman(r)}" for _, r in names.iterrows())
PROMPT_C = f"""You are helping with a cross-cultural colour naming study.
For each colour name below, give the sRGB colour that native speakers of that language would consider
the best example of the name: its most typical instance.

{items}

Reply with one line per name, and nothing else, in exactly this format (R, G, B from 0 to 255):
{names.language.iloc[0]} | {names['name'].iloc[0]} | R, G, B"""
show_prompt(PROMPT_C, AI_PROVIDER)

In [ ]:
reply_c = ai_reply(PROMPT_C, AI_PROVIDER, AI_MODEL, AI_BASE_URL)
pal = read_ai_colours(reply_c, names)
show_ai_palette(pal)

## 3 · Compare with how speakers actually use each name

For every name, the left half of each swatch is the AI's colour and the right half is the Colour Spinner's best example, the colour speakers most often call by that name. The badge gives the CIEDE2000 difference between them.

A reminder of scale: about 1 is just noticeable with the two side by side, about 5 is obvious, and above 10 the two are usually read as different colours.

In [ ]:
#@title Fill in the Spinner's best examples and compare { run: "auto" }
DIVERGES_AT = 10  #@param {type:"slider", min:3, max:30, step:1}

pal = add_best_examples(pal, connect(BACKEND, SHEET))     # looks up only the names that need it
pal["de00"] = [de00(ai, best) if isinstance(ai, str) and isinstance(best, str) else np.nan
               for ai, best in zip(pal.ai_hex, pal.spinner_hex)]
pal = pal.sort_values(["language", "de00"], ascending=[True, False]).reset_index(drop=True)
show_pairs(pal, threshold=DIVERGES_AT)

The map below is the same top-down view of CAM16-UCS colour space used in Part B. Each arrow starts at the AI's colour and ends at the speakers' best example. Long arrows are large errors; arrows that all point the same way for one language suggest a systematic bias, for instance towards English focal colours.

In [ ]:
plot_ai_vs_speakers(pal, LANGUAGES, DIVERGES_AT)
display(error_summary(pal, LANGUAGES, DIVERGES_AT))

### Would speakers call the AI's colour by that name?

The distance above compares the AI's colour with a single point, the name's best example. But a name covers a range of colours, and the Spinner knows how speakers use it across that range. So send each AI colour back to the Spinner's Namer and ask what speakers would call it. If the answer is the name you gave the AI, the colour communicates as intended, even if it isn't the most typical example. If it's a different name, speakers would read the AI's colour as something else.

In [ ]:
spinner = connect(BACKEND, SHEET)
pal["rt_name"], pal["rt_conf"] = None, np.nan
for k, row in pal.iterrows():
    if isinstance(row.ai_hex, str):
        namer = safe(spinner.name, row.ai_hex, row.language)   # what speakers call the AI's colour
        if namer:
            pal.at[k, "rt_name"], pal.at[k, "rt_conf"] = namer["name"], namer["confidence"]
show_round_trip(pal)

## 4 · Ground the palette

An AI colour diverges from observed language use if it is too far from the name's best example, or if speakers would call it by a different name. Choose which rule to apply in the form; by default a colour is replaced if either is true. A replaced colour becomes the Colour Spinner's best example of the name. The distance threshold is the slider in section 3.

In [ ]:
#@title Ground the palette { run: "auto" }
REPLACE_WHEN = "Either"  #@param ["Either", "Too far from the best example", "Speakers would use a different name"]

too_far = [pd.isna(d) or d > DIVERGES_AT for d in pal.de00]
renamed = [isinstance(n, str) and not same_name(n, m, r)
           for n, m, r in zip(pal.rt_name, pal.name, pal.get("romanized", pd.Series(None, index=pal.index)))]
rule = {"Either": [a or b for a, b in zip(too_far, renamed)],
        "Too far from the best example": too_far,
        "Speakers would use a different name": renamed}[REPLACE_WHEN]

# Replace the AI's colour with the speakers' best example wherever the rule says it diverges.
pal["grounded_hex"] = [best if isinstance(best, str) and replace else ai
                       for ai, best, replace in zip(pal.ai_hex, pal.spinner_hex, rule)]
pal["replaced"] = pal.grounded_hex != pal.ai_hex
show_grounded(pal, LANGUAGES)

### Does the grounded palette still work for every viewer?

Grounding fixes the cognitive axis, but best examples of neighbouring names can sit close together. Check the perceptual axis again: for each language's palette, the smallest ΔE00 between any two of its colours under each viewing condition.

In [ ]:
check = closest_pairs(pal, LANGUAGES)      # smallest ΔE00 within each palette, per condition
plot_closest_pairs(check, DIVERGES_AT)
show_grounded_conditions(pal, LANGUAGES)

If a pair falls below the threshold under some condition, you have found a real design tension: the colours speakers expect for two names may be too similar for some viewers. Options to discuss: keep one colour and pick the next-best example of the other from the Spinner's Mapper panel, add a second cue such as a pattern, icon or label, or drop one name from the palette.

## 5 · Export your palette

The files below contain every colour with its name, language, the AI's original and the evidence for any change.

In [ ]:
export_palette(pal, LANGUAGES)

## 6 · Your most striking finding

The cell below picks the name where the AI system's colour was furthest from what speakers mean. You can choose a different one by typing its name in the form. Bring this card to the plenary.

In [ ]:
#@title Finding to share { run: "auto" }
PICK_NAME = ""  #@param {type:"string"}
show_finding(pal, PICK_NAME)

---
*Colour Spinner: Sinkeviciute, Stutters, Ahmed, Koliousis & Mylonas (2026). Benchmark of AI colour naming: Mylonas, Ahmed, Sinkeviciute & Koliousis, CHI 2026. Colour vision deficiency and age simulations from the Colour Spinner. CAM16-UCS after Li et al. (2017).*